# MARV — SmolLM2 tool-calling vindex probe (T4)

Compares a base SmolLM2-135M-Instruct against a tool-call fine-tune of the
same size, plus the larger 1.7B-Instruct (official tool-calling support),
using MARV's gate-KNN + logit-lens + per-feature diff, scoped to fit in a
T4's 16GB.

Runtime: **T4 GPU** (Runtime > Change runtime type > T4).

In [ ]:
!pip install -q transformers accelerate numpy
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

## Load the two 135M checkpoints (base vs. tool-tuned)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

BASE_135M = "HuggingFaceTB/SmolLM2-135M-Instruct"
TUNED_135M = "gvij/SmolLM2-135M-Function-Calling"

base_tok = AutoTokenizer.from_pretrained(BASE_135M)
base_model = AutoModelForCausalLM.from_pretrained(BASE_135M, torch_dtype=torch.float16).to(device).eval()

tuned_tok = AutoTokenizer.from_pretrained(TUNED_135M)
tuned_model = AutoModelForCausalLM.from_pretrained(TUNED_135M, torch_dtype=torch.float16).to(device).eval()

## Extract vindex-lite for both

In [ ]:
from marv.extract import extract

vindex_base = extract(base_model, model_name=BASE_135M)
vindex_tuned = extract(tuned_model, model_name=TUNED_135M)
print(vindex_base.num_layers, "layers,", vindex_base.hidden_size, "hidden")

## Concept probe: label firing features in each checkpoint

Each word is embedded in a short template sentence and run through the live model (a contextual hidden state, not a bare embedding row), then differenced against the template alone -- small transformers have a dominant, roughly prompt-invariant direction ("massive activation"/outlier channel) that otherwise swamps the query regardless of content. What's left after differencing is KNN'd over a *late* layer's gate rows (logit lens only reliably decodes to legible tokens in roughly the last third of the model) and labeled by what each firing feature promotes.

In [ ]:
from marv.toolcall import describe_prompt

PROBE_WORDS = ["weather", "function", "France"]
PROBE_TEMPLATE = "I want to talk about {word}"
PROBE_BASELINE = "I want to talk about"

def concept_probe(vindex, model, tok, label):
    late_layers = sorted(set(round(vindex.num_layers * f) for f in (0.7, 0.85)))
    late_layers = [l for l in late_layers if l < vindex.num_layers]
    print(f"--- {label} ---")
    for word in PROBE_WORDS:
        prompt = PROBE_TEMPLATE.format(word=word)
        hits = describe_prompt(
            vindex, model, tok, prompt, layers=late_layers, k_features=3, k_tokens=3,
            device=device, baseline_prompt=PROBE_BASELINE,
        )
        print(f"'{word}':")
        for layer, features in hits.items():
            for feature_idx, sim, tok_ids, logits in features:
                words_out = tok.batch_decode([[t] for t in tok_ids])
                print(f"  L{layer} f{feature_idx} (sim={sim:.2f}) -> {words_out}")

concept_probe(vindex_base, base_model, base_tok, "base")
concept_probe(vindex_tuned, tuned_model, tuned_tok, "tool-tuned")

## DIFF: which features moved most during tool-call fine-tuning

This is MARV's version-control primitive -- a ranked, sparse list of exactly
which FFN neurons changed, instead of a dense weight delta.

In [ ]:
from marv.diff import diff, most_changed

deltas = diff(vindex_base, vindex_tuned)
top = most_changed(deltas, k=15)
for d in top:
    print(f"L{d.layer} f{d.feature_idx}: gate_cos={d.gate_cos_sim:.3f} down_cos={d.down_cos_sim:.3f} norm_ratio={d.gate_norm_ratio:.2f}")

In [ ]:
# Label what the most-changed features promote, before vs. after
from marv.probe import describe_feature

for d in top[:5]:
    before = describe_feature(vindex_base, d.layer, d.feature_idx, k=3)
    after = describe_feature(vindex_tuned, d.layer, d.feature_idx, k=3)
    print(f"L{d.layer} f{d.feature_idx}")
    print("  before:", base_tok.batch_decode([[t] for t in before[0]]))
    print("  after: ", tuned_tok.batch_decode([[t] for t in after[0]]))

## Tool-call decision point: does the logit-lens answer flip?

Feed a prompt that should trigger a tool call, capture the residual stream at several *late* layers in both checkpoints (logit lens is noisy in early/mid layers), and see what each one "wants to say" at each depth.

In [ ]:
from marv.toolcall import DEFAULT_TOOL_PROMPTS, compare_tool_prompt, hidden_states_at_layers

start = max(1, round(vindex_base.num_layers * 0.66))
probe_layers = list(range(start, vindex_base.num_layers, 3))

for prompt in DEFAULT_TOOL_PROMPTS:
    print("\nprompt:", prompt)
    h_base = hidden_states_at_layers(base_model, base_tok, prompt, probe_layers, device=device)
    h_tuned = hidden_states_at_layers(tuned_model, tuned_tok, prompt, probe_layers, device=device)
    comparison = compare_tool_prompt(vindex_base, vindex_tuned, h_base, h_tuned)
    for layer, result in comparison.items():
        base_words = base_tok.batch_decode([[t] for t, _ in result["base_top"][:3]])
        tuned_words = tuned_tok.batch_decode([[t] for t, _ in result["tuned_top"][:3]])
        print(f"  L{layer}: base={base_words}  tuned={tuned_words}")

## Bring in the 1.7B (official tool-calling) for a size comparison

Note: `diff()` requires matching layer counts, so the 1.7B can't be diffed
directly against the 135M models -- it's probe-only here (DESCRIBE + the
tool-call logit-lens check), which is enough to compare *how* tool-calling
shows up at a larger scale vs. how it looks in the small fine-tune above.

In [ ]:
INSTRUCT_1_7B = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

big_tok = AutoTokenizer.from_pretrained(INSTRUCT_1_7B)
big_model = AutoModelForCausalLM.from_pretrained(INSTRUCT_1_7B, torch_dtype=torch.float16).to(device).eval()
vindex_big = extract(big_model, model_name=INSTRUCT_1_7B)
print(vindex_big.num_layers, "layers,", vindex_big.hidden_size, "hidden")

In [ ]:
from marv.probe import logit_lens

start_big = max(1, round(vindex_big.num_layers * 0.66))
probe_layers_big = list(range(start_big, vindex_big.num_layers, 3))
for prompt in DEFAULT_TOOL_PROMPTS:
    print("\nprompt:", prompt)
    h_big = hidden_states_at_layers(big_model, big_tok, prompt, probe_layers_big, device=device)
    for layer, vec in h_big.items():
        idx, logits = logit_lens(vindex_big, vec, k=3)
        print(f"  L{layer}:", big_tok.batch_decode([[t] for t in idx]))

## Next steps

- Swap `DEFAULT_TOOL_PROMPTS` in `marv/toolcall.py` for the model's real
  tool schema / few-shot format and re-run.
- Cross-reference `most_changed()` feature indices against
  `marv.toolcall.firing_features_at_layer()` on the tool-call prompts, to
  see whether fine-tuning touched the *same* features that are actually
  active when the model decides to call a tool.
- Save any `VindexLite` with `.save("name.npz")` to compare across
  Colab sessions without re-extracting.